# Treinar o modelo "hey chubasa" (openWakeWord)

Este notebook gera o arquivo `hey_chubasa.onnx` que o Chubasa usa para reconhecer a palavra-chave.
É baseado no fluxo oficial de treino do openWakeWord (`notebooks/automatic_model_training.ipynb`
e `openwakeword/train.py` da versão 0.6.0), com estas diferenças:

- **Ambiente Python 3.10 isolado** (criado com `uv`): as dependências de treino (`piper-phonemize`,
  `speechbrain 0.5`, `acoustics`) não instalam no Python atual do Colab.
- **Pronúncia de "chubasa" injetada no dicionário CMU**: o download do DeepPhonemizer que o
  `train.py` usaria para palavras desconhecidas está fora do ar (HTTP 403).
- **Ruído de fundo do AudioSet em parquet** (o `.tar` antigo não existe mais) e sem FMA.
- **Suas gravações** (de `treino/gravar_amostras.py`) entram junto com as amostras sintéticas.
- Sem conversão para tflite (o Chubasa usa ONNX).

**Antes de começar:** Ambiente de execução > Alterar o tipo de ambiente de execução > **GPU (T4)**.
Rode as células em ordem. Tempo total: 1 a 3 horas, a maior parte gerando e processando áudio.
O download maior são as features de fala negativa (17 GB): o disco do Colab comporta.

## 1. Parâmetros

In [ ]:
import json

NOME_MODELO = "hey_chubasa"

# Frases que o TTS (em inglês) vai falar. Em português "Chubasa" soa "shu-BA-za", por isso
# entra também uma grafia fonética. Ouça os exemplos na etapa 6 e ajuste se soar errado.
FRASES = ["hey chubasa", "hey shoo bah zah"]

# Pronúncia (fonemas CMU) das palavras que não estão no dicionário em inglês.
# Toda palavra de FRASES precisa estar no dicionário ou aqui.
PRONUNCIAS = {
    "chubasa": "SH UW0 B AA1 Z AH0",
    "zah": "Z AA1",
}

# Frases parecidas que NÃO devem disparar (além das geradas automaticamente)
NEGATIVAS_EXTRAS = ["hey chuva", "hey shoe", "hey jarvis", "hey siri", "hey bazaar", "hey sugar"]

N_AMOSTRAS = 5000      # clipes sintéticos de treino; 20000+ dá modelo melhor (e demora mais)
N_VALIDACAO = 1000     # clipes sintéticos de teste
PASSOS = 20000         # passos de treino
PESO_NEGATIVO_MAX = 1500       # maior = menos falso disparo, porém menos sensível
FALSOS_POR_HORA_ALVO = 0.2
ARQUIVOS_AUDIOSET = 2  # cada arquivo ~690 MB, ~490 clipes de 10 s de ruído de fundo

# Quantas cópias de cada gravação sua entram no treino (cada cópia recebe ruído/eco diferentes).
# 0 = automático: suas positivas viram ~10% das positivas e suas negativas ~5% das negativas.
COPIAS_POSITIVAS = 0
COPIAS_NEGATIVAS = 0

PARAMETROS = dict(
    NOME_MODELO=NOME_MODELO, FRASES=FRASES, PRONUNCIAS=PRONUNCIAS, NEGATIVAS_EXTRAS=NEGATIVAS_EXTRAS,
    N_AMOSTRAS=N_AMOSTRAS, N_VALIDACAO=N_VALIDACAO, PASSOS=PASSOS, PESO_NEGATIVO_MAX=PESO_NEGATIVO_MAX,
    FALSOS_POR_HORA_ALVO=FALSOS_POR_HORA_ALVO, ARQUIVOS_AUDIOSET=ARQUIVOS_AUDIOSET,
    COPIAS_POSITIVAS=COPIAS_POSITIVAS, COPIAS_NEGATIVAS=COPIAS_NEGATIVAS,
)
json.dump(PARAMETROS, open("/content/parametros.json", "w"), indent=2)
print(open("/content/parametros.json").read())

## 2. Ambiente de treino (Python 3.10)

In [ ]:
%%bash
set -euo pipefail
cd /content
pip install -q uv
uv python install 3.10
[ -x /content/oww/bin/python ] || uv venv -q --python 3.10 /content/oww
uv pip install -q --python /content/oww/bin/python \
  "torch==2.1.2" "torchaudio==2.1.2" "numpy==1.26.4" "scipy==1.11.4" \
  "openwakeword==0.6.0" "onnxruntime==1.16.3" "scikit-learn==1.3.2" \
  "mutagen==1.47.0" "torchinfo==1.8.0" "torchmetrics==1.2.0" "speechbrain==0.5.14" \
  "audiomentations==0.33.0" "torch-audiomentations==0.11.0" "acoustics==0.2.6" \
  "pronouncing==0.2.0" "piper-phonemize==1.1.0" "webrtcvad==2.0.10" \
  "huggingface_hub==0.23.5" "pyarrow==15.0.2" "soundfile==0.12.1" "setuptools<81" \
  pyyaml tqdm requests

# Gerador de amostras sintéticas (TTS Piper) na versão que o openWakeWord 0.6.0 espera
[ -d piper-sample-generator ] || git clone -q --depth 1 --branch v2.0.0 https://github.com/rhasspy/piper-sample-generator
[ -f piper-sample-generator/models/en_US-libritts_r-medium.pt ] || wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt \
  https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt

# Modelos de pré-processamento do openWakeWord (melspectrogram/embedding)
/content/oww/bin/python -c "import openwakeword.utils as u; u.download_models(['hey_jarvis'])"
/content/oww/bin/python -c "import torch; print('torch', torch.__version__, '| GPU:', torch.cuda.is_available())"

## 3. Dados de apoio: ecos de sala, ruído de fundo e fala negativa

In [ ]:
%%writefile /content/baixar_dados.py
import io
import json
from math import gcd
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq
import scipy.io.wavfile
import soundfile as sf
from huggingface_hub import hf_hub_download, snapshot_download
from scipy.signal import resample_poly
from tqdm import tqdm

P = json.load(open("/content/parametros.json"))
BASE = Path("/content")

# Respostas ao impulso de salas reais (MIT), já em 16 kHz
rirs = BASE / "mit_rirs"
if not rirs.exists():
    pasta = snapshot_download("davidscripka/MIT_environmental_impulse_responses", repo_type="dataset",
                              allow_patterns=["16khz/*.wav"], local_dir=BASE / "mit_rirs_hf")
    (Path(pasta) / "16khz").rename(rirs)
print("RIRs:", len(list(rirs.glob("*.wav"))))

# Ruído de fundo: clipes do AudioSet (parquet com FLAC embutido), convertidos para WAV 16 kHz mono
fundo = BASE / "audioset_16k"
fundo.mkdir(exist_ok=True)
for n in range(P["ARQUIVOS_AUDIOSET"]):
    marcador = fundo / f".feito_{n:02d}"
    if marcador.exists():
        continue
    arquivo = hf_hub_download("agkphysics/AudioSet", f"data/bal_train/{n:02d}.parquet", repo_type="dataset",
                              local_dir=BASE / "audioset_tmp")
    tabela = pq.read_table(arquivo, columns=["audio"])
    for item in tqdm(tabela.column("audio").to_pylist(), desc=f"AudioSet {n:02d}"):
        try:
            dados, taxa = sf.read(io.BytesIO(item["bytes"]), dtype="float32", always_2d=True)
        except RuntimeError:
            continue  # clipe corrompido: pula
        mono = dados.mean(axis=1)
        g = gcd(taxa, 16000)
        mono = resample_poly(mono, 16000 // g, taxa // g)
        nome = Path(item["path"]).stem + ".wav"
        scipy.io.wavfile.write(fundo / nome, 16000, (np.clip(mono, -1, 1) * 32767).astype(np.int16))
    Path(arquivo).unlink(missing_ok=True)
    marcador.touch()
print("Clipes de fundo:", len(list(fundo.glob("*.wav"))))

In [ ]:
!/content/oww/bin/python /content/baixar_dados.py
# Features pré-calculadas de ~2000 h de fala/ruído (negativas) e conjunto de validação de falsos disparos
!cd /content && wget -q -c https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!cd /content && wget -q -c https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy
!ls -lh /content/*.npy

## 4. Suas gravações (opcional, recomendado)

Envie o `amostras.zip` gerado por `python treino/gravar_amostras.py` no seu PC.
Se não tiver, pule esta célula: o modelo é treinado só com vozes sintéticas.

In [ ]:
import shutil, zipfile
from pathlib import Path
from google.colab import files

enviados = files.upload()
destino = Path("/content/amostras")
shutil.rmtree(destino, ignore_errors=True)
for nome in enviados:
    with zipfile.ZipFile(nome) as z:
        z.extractall(destino)
print("positivas:", len(list(destino.glob("positivas/*.wav"))), "| negativas:", len(list(destino.glob("negativas/*.wav"))))

## 5. Configuração do treino

In [ ]:
import json
import yaml

P = json.load(open("/content/parametros.json"))
config = {
    "model_name": P["NOME_MODELO"],
    "target_phrase": P["FRASES"],
    "custom_negative_phrases": P["NEGATIVAS_EXTRAS"],
    "n_samples": P["N_AMOSTRAS"],
    "n_samples_val": P["N_VALIDACAO"],
    "tts_batch_size": 50,
    "augmentation_batch_size": 16,
    "augmentation_rounds": 1,
    "piper_sample_generator_path": "/content/piper-sample-generator",
    "output_dir": "/content/modelo",
    "rir_paths": ["/content/mit_rirs"],
    "background_paths": ["/content/audioset_16k"],
    "background_paths_duplication_rate": [1],
    "false_positive_validation_data_path": "/content/validation_set_features.npy",
    "feature_data_files": {"ACAV100M_sample": "/content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy"},
    "batch_n_per_class": {"ACAV100M_sample": 1024, "adversarial_negative": 50, "positive": 50},
    "model_type": "dnn",
    "layer_size": 32,
    "steps": P["PASSOS"],
    "max_negative_weight": P["PESO_NEGATIVO_MAX"],
    "target_false_positives_per_hour": P["FALSOS_POR_HORA_ALVO"],
}
yaml.dump(config, open("/content/treino.yaml", "w"), sort_keys=False, allow_unicode=True)
print(open("/content/treino.yaml").read())

In [ ]:
%%writefile /content/treinar.py
"""Roda openwakeword.train com a pronúncia de "chubasa" injetada no dicionário CMU."""
import json
import os
import runpy
import sys

import pronouncing

P = json.load(open("/content/parametros.json"))
pronouncing.init_cmu()
for palavra, fones in P["PRONUNCIAS"].items():
    pronouncing.lookup[palavra.lower()] = [fones]

faltando = sorted({w for frase in P["FRASES"] for w in frase.lower().split() if not pronouncing.phones_for_word(w)})
if faltando:
    sys.exit(f"Sem pronúncia para: {', '.join(faltando)}. Adicione em PRONUNCIAS (etapa 1).")

os.chdir("/content")
sys.argv = ["train.py", "--training_config", "/content/treino.yaml", *sys.argv[1:]]
try:
    runpy.run_module("openwakeword.train", run_name="__main__", alter_sys=True)
except ModuleNotFoundError as erro:
    onnx = f"/content/modelo/{P['NOME_MODELO']}.onnx"
    # O train.py tenta converter para tflite no fim (precisa de TensorFlow); o ONNX já foi salvo.
    if "--train_model" in sys.argv and erro.name in {"onnx", "onnx_tf", "tensorflow"} and os.path.exists(onnx):
        print(f"Modelo ONNX salvo em {onnx} (conversão para tflite pulada: não é usada).")
    else:
        raise

## 6. Gerar vozes sintéticas

In [ ]:
!/content/oww/bin/python /content/treinar.py --generate_clips

In [ ]:
# Ouça alguns exemplos: se "chubasa" estiver soando errado, ajuste FRASES/PRONUNCIAS na etapa 1,
# apague /content/modelo e gere de novo.
import random
from pathlib import Path
from IPython.display import Audio, display

P = json.load(open("/content/parametros.json"))
clipes = list(Path(f"/content/modelo/{P['NOME_MODELO']}/positive_train").glob("*.wav"))
for clipe in random.sample(clipes, min(4, len(clipes))):
    display(Audio(str(clipe)))

## 7. Juntar suas gravações às sintéticas

In [ ]:
import shutil
from pathlib import Path

P = json.load(open("/content/parametros.json"))
pasta = Path(f"/content/modelo/{P['NOME_MODELO']}")
suas = {"positivas": sorted(Path("/content/amostras").glob("positivas/*.wav")),
        "negativas": sorted(Path("/content/amostras").glob("negativas/*.wav"))}

def copias(n_suas, n_sinteticas, fixo, fracao):
    if n_suas == 0:
        return 0
    return fixo if fixo > 0 else max(1, round(fracao * n_sinteticas / n_suas))

for tipo, destino, fixo, fracao in (("positivas", "positive_train", P["COPIAS_POSITIVAS"], 0.10),
                                    ("negativas", "negative_train", P["COPIAS_NEGATIVAS"], 0.05)):
    alvo = pasta / destino
    for antigo in alvo.glob("minha_voz_*.wav"):
        antigo.unlink()
    k = copias(len(suas[tipo]), len(list(alvo.glob("*.wav"))), fixo, fracao)
    for wav in suas[tipo]:
        for i in range(k):
            shutil.copy(wav, alvo / f"minha_voz_{wav.stem}_{i:03d}.wav")
    print(f"{tipo}: {len(suas[tipo])} gravações x {k} cópias -> {destino} ({len(list(alvo.glob('*.wav')))} clipes)")

## 8. Aumentar dados (ruído, eco) e calcular features

In [ ]:
!/content/oww/bin/python /content/treinar.py --augment_clips --overwrite

## 9. Treinar

In [ ]:
!/content/oww/bin/python /content/treinar.py --train_model

## 10. Conferir e baixar

In [ ]:
%%writefile /content/conferir.py
import json
from pathlib import Path

import numpy as np
from openwakeword.model import Model

P = json.load(open("/content/parametros.json"))
onnx = f"/content/modelo/{P['NOME_MODELO']}.onnx"
modelo = Model(wakeword_models=[onnx], inference_framework="onnx")
for tipo in ("positivas", "negativas"):
    wavs = sorted(Path("/content/amostras").glob(f"{tipo}/*.wav"))
    if not wavs:
        continue
    picos = [max(p[P["NOME_MODELO"]] for p in modelo.predict_clip(str(w))) for w in wavs]
    print(f"Suas {tipo}: score máximo mediano {np.median(picos):.2f} "
          f"| acima de 0.5: {sum(s >= 0.5 for s in picos)}/{len(picos)}")

In [ ]:
!/content/oww/bin/python /content/conferir.py
import json
from google.colab import files
files.download(f"/content/modelo/{json.load(open('/content/parametros.json'))['NOME_MODELO']}.onnx")

## Depois

1. Copie o `hey_chubasa.onnx` baixado para `models\hey_chubasa.onnx` na pasta do Chubasa.
2. Rode `python -m chubasa escutar` e diga "hey chubasa" algumas vezes.
3. Ajuste `WAKE_THRESHOLD` no `.env` olhando os scores do log: suba se disparar à toa, desça se não ouvir.

O score das suas gravações na etapa 10 é otimista (elas entraram no treino); o teste que vale é ao vivo.